# ProteinLens — Day 2: MobileNetV2 (India-optimised)

**Platform:** Kaggle — GPU T4 x2  
**Dataset:** Food-101 via TFDS  
**Classes:** 20 foods eaten daily in Indian households  
**Target:** 85%+ validation accuracy

### Key fixes vs previous runs
- Reduced steps/epoch so training never gets stuck
- Indian food classes: chicken curry, samosa, momos, omelette, fried rice etc.
- Augment on `[0,255]` THEN normalise — correct order
- Separate `eval_ds` with no `.repeat()` for clean evaluation

In [ ]:
# ── CELL 1: GPU setup ────────────────────────────────────────
import os, json
import numpy as np
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Suppress verbose CUDA logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print(f'TensorFlow : {tf.__version__}')
print(f'GPUs       : {len(gpus)}')
for g in gpus:
    print(f'  {g}')

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# ── CELL 2: Install ──────────────────────────────────────────
!pip install tensorflow-datasets scikit-learn seaborn -q
print('Done')

In [ ]:
# ── CELL 3: Imports ──────────────────────────────────────────
import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from sklearn.metrics import classification_report, confusion_matrix
print('Imports OK')

In [ ]:
# ── CELL 4: Configuration ────────────────────────────────────
IMG_SIZE      = 224
BATCH_SIZE    = 64     # 64 on Kaggle T4x2 — faster epochs, won't get stuck
NUM_CLASSES   = 20
PHASE1_EPOCHS = 8      # more epochs Phase 1 for better head training
PHASE2_EPOCHS = 20     # more epochs Phase 2 for fine-tuning
PHASE1_LR     = 1e-3
PHASE2_LR     = 5e-5   # lower LR for Phase 2 — more stable
UNFREEZE_FROM = 120    # unfreeze fewer layers — more stable fine-tuning
OUTPUT_DIR    = '/kaggle/working'

# ── 20 Indian-household daily foods — all in Food-101 ────────
TARGET_CLASSES = [
    'chicken_curry',     # Indian staple
    'samosa',            # Indian street food
    'omelette',          # Daily Indian breakfast
    'eggs_benedict',     # Egg dish
    'french_toast',      # Bread+egg breakfast
    'fried_rice',        # Daily across all India
    'dumplings',         # Momos — hugely popular
    'spring_rolls',      # Street food
    'hot_and_sour_soup', # Chinese-Indian popular
    'pad_thai',          # Noodles — common
    'chicken_wings',     # Chicken — high protein
    'club_sandwich',     # Chicken sandwich
    'hamburger',         # Burger — very common now
    'pizza',             # Extremely common in India
    'grilled_salmon',    # Fish — high protein
    'hummus',            # Chickpea protein
    'falafel',           # Chickpea fritter
    'pancakes',          # Common breakfast
    'bibimbap',          # Rice+egg bowl
    'steak',             # Beef/paneer steak
]

CLASS_DISPLAY = {
    'chicken_curry'    : 'Chicken Curry',
    'samosa'           : 'Samosa',
    'omelette'         : 'Omelette',
    'eggs_benedict'    : 'Egg',
    'french_toast'     : 'French Toast',
    'fried_rice'       : 'Fried Rice',
    'dumplings'        : 'Momos',
    'spring_rolls'     : 'Spring Rolls',
    'hot_and_sour_soup': 'Hot & Sour Soup',
    'pad_thai'         : 'Noodles',
    'chicken_wings'    : 'Chicken',
    'club_sandwich'    : 'Sandwich',
    'hamburger'        : 'Burger',
    'pizza'            : 'Pizza',
    'grilled_salmon'   : 'Fish',
    'hummus'           : 'Hummus',
    'falafel'          : 'Falafel',
    'pancakes'         : 'Pancakes',
    'bibimbap'         : 'Rice Bowl',
    'steak'            : 'Steak',
}

PROTEIN_DATA = {
    'chicken_curry'    : {'protein_g': 25.0, 'calories': 150},
    'samosa'           : {'protein_g': 6.0,  'calories': 262},
    'omelette'         : {'protein_g': 10.9, 'calories': 154},
    'eggs_benedict'    : {'protein_g': 12.6, 'calories': 155},
    'french_toast'     : {'protein_g': 8.5,  'calories': 229},
    'fried_rice'       : {'protein_g': 4.4,  'calories': 163},
    'dumplings'        : {'protein_g': 7.0,  'calories': 180},
    'spring_rolls'     : {'protein_g': 4.0,  'calories': 153},
    'hot_and_sour_soup': {'protein_g': 3.8,  'calories': 71},
    'pad_thai'         : {'protein_g': 10.0, 'calories': 181},
    'chicken_wings'    : {'protein_g': 27.0, 'calories': 203},
    'club_sandwich'    : {'protein_g': 18.0, 'calories': 290},
    'hamburger'        : {'protein_g': 20.3, 'calories': 295},
    'pizza'            : {'protein_g': 11.0, 'calories': 266},
    'grilled_salmon'   : {'protein_g': 25.4, 'calories': 208},
    'hummus'           : {'protein_g': 7.9,  'calories': 177},
    'falafel'          : {'protein_g': 13.3, 'calories': 333},
    'pancakes'         : {'protein_g': 5.7,  'calories': 227},
    'bibimbap'         : {'protein_g': 12.0, 'calories': 190},
    'steak'            : {'protein_g': 26.1, 'calories': 271},
}

# Steps — with BATCH_SIZE=64, epochs are faster and won't get stuck
n_train          = 750 * NUM_CLASSES    # 15,000
n_val            = 250 * NUM_CLASSES    # 5,000
STEPS_PER_EPOCH  = n_train // BATCH_SIZE   # 234 steps — fast epochs
VALIDATION_STEPS = n_val   // BATCH_SIZE   # 78 steps

print(f'Classes         : {NUM_CLASSES}')
print(f'Batch size      : {BATCH_SIZE} (larger = faster epochs)')
print(f'Steps/epoch     : {STEPS_PER_EPOCH} (was 468 before — now 2x faster)')
print(f'Validation steps: {VALIDATION_STEPS}')
print(f'\nIndian food classes:')
for i, c in enumerate(TARGET_CLASSES):
    p = PROTEIN_DATA[c]['protein_g']
    print(f'  {i:02d}. {c:<22} -> "{CLASS_DISPLAY[c]}"  ({p}g protein)')

In [ ]:
# ── CELL 5: Load Food-101 via TFDS ──────────────────────────
print('Loading Food-101 (cached if already downloaded)...')

(ds_train_raw, ds_val_raw), ds_info = tfds.load(
    'food101',
    split=['train', 'validation'],
    with_info=True,
    as_supervised=True,
    shuffle_files=False,
)

all_classes  = ds_info.features['label'].names
class_to_idx = {name: idx for idx, name in enumerate(all_classes)}

# Verify all target classes exist
missing = [c for c in TARGET_CLASSES if c not in class_to_idx]
if missing:
    raise ValueError(f'Missing from Food-101: {missing}')
print(f'All {NUM_CLASSES} classes verified in Food-101')

# Build lookup tensor: food101_idx -> new 0..19  (-1 = discard)
target_old_indices = [class_to_idx[c] for c in TARGET_CLASSES]
lookup_table       = [-1] * len(all_classes)
for new_idx, old_idx in enumerate(target_old_indices):
    lookup_table[old_idx] = new_idx
lookup_tensor = tf.constant(lookup_table, dtype=tf.int32)

print('Label mapping:')
for i, cls in enumerate(TARGET_CLASSES):
    print(f'  {i:02d}. {cls:<22} food101_idx:{target_old_indices[i]:>3}')

In [ ]:
# ── CELL 6: Data pipeline ───────────────────────────────────
# CRITICAL: augment on [0,255] THEN normalise to [-1,1]

AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='aug')

def preprocess_train(image, label):
    new_label = lookup_tensor[tf.cast(label, tf.int32)]
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)                                  # [0,255]
    image = augment(image, training=True)                               # augment FIRST
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image) # [-1,1] LAST
    return image, new_label

def preprocess_val(image, label):
    new_label = lookup_tensor[tf.cast(label, tf.int32)]
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, new_label

def is_target(image, label):
    return tf.not_equal(label, -1)

train_ds = (
    ds_train_raw
    .map(preprocess_train, num_parallel_calls=AUTOTUNE)
    .filter(is_target)
    .shuffle(2000, seed=42)
    .batch(BATCH_SIZE, drop_remainder=True)
    .repeat()
    .prefetch(AUTOTUNE)
)

val_ds = (
    ds_val_raw
    .map(preprocess_val, num_parallel_calls=AUTOTUNE)
    .filter(is_target)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(AUTOTUNE)
)

# Clean eval — no repeat, no augmentation
eval_ds = (
    ds_val_raw
    .map(preprocess_val, num_parallel_calls=AUTOTUNE)
    .filter(is_target)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f'Train : ~{n_train:,} samples -> {STEPS_PER_EPOCH} steps/epoch')
print(f'Val   : ~{n_val:,} samples  -> {VALIDATION_STEPS} steps')
print(f'Each epoch takes ~3-5 mins on T4 x2 (was 7-8 mins before)')

# Sanity check
for imgs, lbls in train_ds.take(1):
    pmin   = round(float(imgs.numpy().min()), 2)
    pmax   = round(float(imgs.numpy().max()), 2)
    unique = sorted(set(lbls.numpy().tolist()))
    ok_p   = pmin >= -1.1 and pmax <= 1.1
    ok_l   = all(0 <= l <= 19 for l in unique)
    print(f'\nSanity: shape={imgs.shape} pixels=[{pmin},{pmax}] labels={unique[:6]}...')
    if not ok_p or not ok_l:
        raise ValueError('FAILED — stop here')
    print('ALL GOOD')

In [ ]:
# ── CELL 7: Build model ─────────────────────────────────────
def build_model(num_classes, trainable_base=False):
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = trainable_base

    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation='relu',
                           kernel_regularizer=l2(1e-4))(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(256, activation='relu',
                           kernel_regularizer=l2(1e-4))(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='ProteinLens'), base

model, base_model = build_model(NUM_CLASSES, trainable_base=False)

trainable = sum(np.prod(v.shape) for v in model.trainable_weights)
total     = sum(np.prod(v.shape) for v in model.weights)
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
model.summary()

In [ ]:
# ── CELL 8: Phase 1 — train head (base frozen) ──────────────
# With BATCH_SIZE=64 each epoch takes ~3-4 mins vs 7 mins before
# Expected: Epoch 1 val_accuracy ~65-75% (already saw 71% before)

model.compile(
    optimizer=keras.optimizers.Adam(PHASE1_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb1 = [
    EarlyStopping(monitor='val_accuracy', patience=3,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, min_lr=1e-5, verbose=1),
    ModelCheckpoint(f'{OUTPUT_DIR}/best_phase1.keras',
                    monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

print(f'Phase 1: {PHASE1_EPOCHS} epochs x {STEPS_PER_EPOCH} steps')
print(f'Estimated time: ~{PHASE1_EPOCHS * 4} mins')
print('='*50)

history1 = model.fit(
    train_ds,
    epochs=PHASE1_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VALIDATION_STEPS,
    callbacks=cb1,
    verbose=1
)

p1_best = max(history1.history['val_accuracy'])
print(f'\nPhase 1 best: {p1_best*100:.1f}%')

In [ ]:
# ── CELL 9: Phase 2 — fine-tune ─────────────────────────────
print(f'Unfreezing from layer {UNFREEZE_FROM} / {len(base_model.layers)}')

base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen: {unfrozen} layers')

model.compile(
    optimizer=keras.optimizers.Adam(PHASE2_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb2 = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{OUTPUT_DIR}/best_model.keras',
                    monitor='val_accuracy',
                    save_best_only=True, verbose=1),
]

print(f'\nPhase 2: up to {PHASE2_EPOCHS} epochs x {STEPS_PER_EPOCH} steps')
print(f'Early stopping if no improvement for 5 epochs')
print('='*50)

history2 = model.fit(
    train_ds,
    epochs=PHASE2_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VALIDATION_STEPS,
    callbacks=cb2,
    verbose=1
)

p2_best = max(history2.history['val_accuracy'])
print(f'\nPhase 2 best: {p2_best*100:.1f}%')

In [ ]:
# ── CELL 10: Training curves ────────────────────────────────
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
ep       = range(1, len(acc) + 1)
p2s      = len(history1.history['accuracy']) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ProteinLens — MobileNetV2 India-optimised', fontweight='bold')

ax1.plot(ep, acc,     '#27ae60', label='Train', linewidth=2)
ax1.plot(ep, val_acc, '#e74c3c', label='Val',   linewidth=2)
ax1.axvline(p2s, color='gray', linestyle='--', alpha=0.7, label=f'Fine-tune ep{p2s}')
ax1.axhline(0.85, color='#2980b9', linestyle=':', label='85% target')
ax1.set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0,1))
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, loss,     '#27ae60', label='Train', linewidth=2)
ax2.plot(ep, val_loss, '#e74c3c', label='Val',   linewidth=2)
ax2.axvline(p2s, color='gray', linestyle='--', alpha=0.7)
ax2.set(title='Loss', xlabel='Epoch', ylabel='Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ── CELL 11: Evaluation ─────────────────────────────────────
print('Evaluating on clean validation set...')
y_true, y_pred = [], []
for imgs, lbls in eval_ds:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(lbls.numpy())

y_true      = np.array(y_true)
y_pred      = np.array(y_pred)
overall_acc = np.mean(y_true == y_pred)
short_names = [CLASS_DISPLAY[c] for c in TARGET_CLASSES]

print(f'\nFinal Validation Accuracy: {overall_acc*100:.2f}%')
print(classification_report(y_true, y_pred,
                             target_names=short_names, digits=3))

# Confusion matrix
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=short_names, yticklabels=short_names,
            linewidths=0.4, ax=ax)
ax.set_title('ProteinLens — Confusion Matrix (India)', fontsize=14, fontweight='bold')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

In [ ]:
# ── CELL 12: Save model + label map ─────────────────────────
model.save(f'{OUTPUT_DIR}/ProteinLens_model.keras')
print('Saved: ProteinLens_model.keras')

label_map = {
    'target_classes' : TARGET_CLASSES,
    'display_names'  : CLASS_DISPLAY,
    'protein_data'   : PROTEIN_DATA,
    'idx_to_class'   : {str(i): c for i, c in enumerate(TARGET_CLASSES)},
    'idx_to_display' : {str(i): CLASS_DISPLAY[c]
                        for i, c in enumerate(TARGET_CLASSES)},
    'idx_to_protein' : {str(i): PROTEIN_DATA[c]
                        for i, c in enumerate(TARGET_CLASSES)},
    'model_info': {
        'architecture'  : 'MobileNetV2',
        'input_size'    : IMG_SIZE,
        'num_classes'   : NUM_CLASSES,
        'val_accuracy'  : round(float(overall_acc), 4),
        'preprocessing' : 'mobilenet_v2 [-1,1]',
        'optimised_for' : 'India'
    }
}

with open(f'{OUTPUT_DIR}/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print('Saved: label_map.json')

print(f'\n==============================')
print(f'  Phase 1 best : {max(history1.history["val_accuracy"])*100:.1f}%')
print(f'  Phase 2 best : {max(history2.history["val_accuracy"])*100:.1f}%')
print(f'  Final val acc: {overall_acc*100:.2f}%')
print(f'  85% target   : {"HIT" if overall_acc >= 0.85 else "close — see below"}')
print(f'==============================')
print('\nDownload all files from the Output tab on the right panel')

## After training — Git push

Download 4 files from Kaggle Output tab, then run on your Mac:

```bash
cd ~/ProteinLens
cp ~/Downloads/label_map.json        ml/
cp ~/Downloads/training_curves.png   assets/
cp ~/Downloads/confusion_matrix.png  assets/
cp ~/Downloads/ProteinLens_model.keras ml/
echo 'ml/*.keras' >> .gitignore

git add ml/train.ipynb ml/label_map.json assets/ .gitignore
git commit -m "Day 2: MobileNetV2 India-optimised — XX% val accuracy"
git push origin main
```